In [1]:
# CELL 1
import pandas as pd
import numpy as np
import gc
import warnings
warnings.filterwarnings('ignore')

BASE = '/kaggle/input/notebooks/luminhanh/ba-model-prep-data'
print("🚀 CELL 1: NẠP DỮ LIỆU GỐC & RÈN VŨ KHÍ HẠNG NẶNG (LAGS/ROLLING/PROMO)...")

# 1. Load Data (🔥 Lùi mốc thời gian về 01/01/2017 để lấy đà tính Rolling 60 ngày)
print("   -> Đang đọc train_clean.parquet từ 2017-01-01...")
train_df = pd.read_parquet(f'{BASE}/train_clean.parquet')
train_df = train_df[train_df['date'] >= '2017-01-01'].copy()
train_df['item_nbr'] = train_df['item_nbr'].astype(int)
train_df = train_df.sort_values(['store_nbr', 'item_nbr', 'date']).reset_index(drop=True)

# 2. Tạo Groupby objects để xử lý Time-Series
grp_sales = train_df.groupby(['store_nbr', 'item_nbr'], observed=True)['unit_sales']
grp_promo = train_df.groupby(['store_nbr', 'item_nbr'], observed=True)['onpromotion']

# 3. Tạo Target & Promo Future (Dự báo 1-16 ngày)
print("   -> Đang Time-shifting 16 ngày tương lai...")
for i in range(1, 17):
    train_df[f'target_{i}']       = grp_sales.shift(-i).astype('float32')
    train_df[f'promo_future_{i}'] = grp_promo.shift(-i).fillna(0).astype('int8')

# 4. [KHO VŨ KHÍ] Feature Engineering Nâng Cao
print("   -> ⚔️ Đang rèn vũ khí: Lags (Ngắn/Chu kỳ), Rolling Means và Promo...")

# 4.1 Lags ngắn hạn (Tương tác gần)
train_df['lag_0'] = train_df['unit_sales'].astype('float32')
for i in [1, 2, 3]: 
    train_df[f'lag_{i}'] = grp_sales.shift(i).astype('float32')

# 4.2 Lags chu kỳ (Bắt Trend mua sắm theo Tuần)
for i in [7, 14, 21, 28]: 
    train_df[f'lag_{i}'] = grp_sales.shift(i).astype('float32')

# 4.3 Rolling Means (Mức nền vĩ mô)
windows = [7, 14, 30, 60]
for w in windows:
    train_df[f'rmean_{w}'] = grp_sales.rolling(w).mean().reset_index(level=[0, 1], drop=True).astype('float32')

# 4.4 Lịch sử Khuyến mãi (Độ "nóng" của item)
train_df['promo_streak_3d'] = grp_promo.rolling(3).sum().reset_index(level=[0, 1], drop=True).astype('float32').fillna(0).astype('int8')
train_df['promo_streak_7d'] = grp_promo.rolling(7).sum().reset_index(level=[0, 1], drop=True).astype('float32').fillna(0).astype('int8')

# Dọn dẹp RAM ngay lập tức
del grp_sales, grp_promo; gc.collect()
print(f"✅ HOÀN TẤT CELL 1! Shape hiện tại: {train_df.shape}")

🚀 CELL 1: NẠP DỮ LIỆU GỐC & RÈN VŨ KHÍ HẠNG NẶNG (LAGS/ROLLING/PROMO)...
   -> Đang đọc train_clean.parquet từ 2017-01-01...
   -> Đang Time-shifting 16 ngày tương lai...
   -> ⚔️ Đang rèn vũ khí: Lags (Ngắn/Chu kỳ), Rolling Means và Promo...
✅ HOÀN TẤT CELL 1! Shape hiện tại: (35688322, 100)


In [2]:
# CELL 2
print("🚀 CELL 2: TẠO TẬP TEST BASE (CHỐT TẠI 15/08)...")

# Trích xuất nền tảng Test
X_test_base = train_df[train_df['date'] == '2017-08-15'].copy()

# Xóa các cột target/sales không cần thiết
cols_to_drop = [f'target_{i}' for i in range(1, 17)] + ['unit_sales']
X_test_base.drop(columns=cols_to_drop, inplace=True, errors='ignore')

# Đồng bộ promo_future từ test_clean
print("   -> Đang đồng bộ thông tin Promo từ test_clean...")
test_raw = pd.read_parquet(f'{BASE}/test_clean.parquet')
test_raw['store_nbr'] = test_raw['store_nbr'].astype(str)
test_raw['item_nbr']  = test_raw['item_nbr'].astype(float).fillna(-1).astype(int)

test_promo = test_raw.pivot_table(index=['store_nbr', 'item_nbr'], columns='date', values='onpromotion', aggfunc='max')
test_promo.columns = [f'promo_future_{i}' for i in range(1, 17)]
test_promo = test_promo.reset_index()

X_test_base.drop(columns=[f'promo_future_{i}' for i in range(1, 17)], inplace=True, errors='ignore')
X_test_base = X_test_base.merge(test_promo, on=['store_nbr', 'item_nbr'], how='left')

for i in range(1, 17):
    X_test_base[f'promo_future_{i}'] = X_test_base[f'promo_future_{i}'].fillna(0).astype('int8')

# Dọn dẹp (Ép category đã được gom toàn bộ về Cell 3 để tránh thừa thãi)
X_test_base = X_test_base.drop_duplicates(subset=['store_nbr', 'item_nbr'], keep='last')

del test_promo, test_raw; gc.collect()
print(f"✅ X_test_base đã sẵn sàng! Shape: {X_test_base.shape}")

🚀 CELL 2: TẠO TẬP TEST BASE (CHỐT TẠI 15/08)...
   -> Đang đồng bộ thông tin Promo từ test_clean...
✅ X_test_base đã sẵn sàng! Shape: (167515, 83)


In [3]:
# CELL 3
import xgboost as xgb
import optuna
from tqdm.notebook import tqdm

print("🚀 CELL 3: TIỀN XỬ LÝ, TUNING OPTUNA & HUẤN LUYỆN SEED AVERAGING...")

# 1. Dọn dẹp NaN (Bắt buộc đồng bộ Train/Test)
print("   -> Đang dọn dẹp Inf và NaN...")
numeric_cols = train_df.select_dtypes(include=[np.number]).columns
for col in numeric_cols:
    col_type = train_df[col].dtype
    train_df[col] = train_df[col].replace([np.inf, -np.inf], np.nan).fillna(0).astype(col_type)
    if col in X_test_base.columns:
        X_test_base[col] = X_test_base[col].replace([np.inf, -np.inf], np.nan).fillna(0).astype(col_type)

# 2. Cắt Ranh giới Train / Valid
print("   -> Cắt ranh giới Validation (16/07 - 30/07)...")
# Khởi điểm dữ liệu train đẩy lên 01/04 để vứt bỏ đoạn rác NaN do shift 60 ngày
train_mask = (train_df['date'] >= '2017-04-01') & (train_df['date'] <= '2017-07-15')
valid_mask = (train_df['date'] > '2017-07-15') & (train_df['date'] <= '2017-07-30')

df_train = train_df[train_mask].copy()
df_valid = train_df[valid_mask].copy()

cols_to_drop = ['date', 'unit_sales', 'id']
df_train.drop(columns=[c for c in cols_to_drop if c in df_train.columns], inplace=True, errors='ignore')
df_valid.drop(columns=[c for c in cols_to_drop if c in df_valid.columns], inplace=True, errors='ignore')
del train_df; gc.collect()

# 3. Tính Trọng số (Weights) - Đã bọc khiên an toàn KeyError
if 'perishable' in df_train.columns:
    weight_train = np.where(df_train['perishable'].astype(int) == 1, 1.25, 1.0).astype(np.float32)
    weight_valid = np.where(df_valid['perishable'].astype(int) == 1, 1.25, 1.0).astype(np.float32)
else:
    weight_train = np.ones(len(df_train), dtype=np.float32)
    weight_valid = np.ones(len(df_valid), dtype=np.float32)

# 4. Cấu hình Cột & Ép kiểu Category (Làm tập trung 1 lần cho cả Train/Valid/Test)
safe_cat_features = ['store_nbr', 'item_nbr', 'city', 'state', 'type', 'cluster', 'family', 'class']
for col in safe_cat_features:
    if col in df_train.columns:
        df_train[col] = df_train[col].astype(str).astype('category')
        df_valid[col] = df_valid[col].astype(str).astype('category')
        if col in X_test_base.columns:
            X_test_base[col] = X_test_base[col].astype(str).astype('category')

exclude_cols = set(safe_cat_features) | {'date', 'unit_sales', 'id', 'perishable'} | \
               {f'target_{i}' for i in range(1, 17)} | {f'promo_future_{i}' for i in range(1, 17)}

base_features = [c for c in df_train.columns if c not in exclude_cols]
if 'perishable' in df_train.columns: base_features.append('perishable')

print(f"   -> 🔍 Số feature sẽ đưa vào Model: {len(base_features)} (Kho vũ khí đã sẵn sàng)")

# ---------------------------------------------------------
# 5. TÌM THAM SỐ TỐI ƯU BẰNG OPTUNA
# ---------------------------------------------------------
print("\n🔍 BẮT ĐẦU TÌM KIẾM THAM SỐ TỐI ƯU BẰNG OPTUNA (5 Trials/Chặng để tiết kiệm thời gian)...")

def run_optuna_for_horizon(day_idx, depth_range, lr_range, lambda_range, n_trials=5):
    print(f"\n🎯 Đang Tune cho đại diện Ngày {day_idx}...")
    tune_features = base_features + safe_cat_features + [f'promo_future_{day_idx}']
    
    dtrain_tune = xgb.DMatrix(df_train[tune_features], label=df_train[f'target_{day_idx}'].fillna(0), weight=weight_train, enable_categorical=True)
    dvalid_tune = xgb.DMatrix(df_valid[tune_features], label=df_valid[f'target_{day_idx}'].fillna(0), weight=weight_valid, enable_categorical=True)
    
    def objective(trial):
        params = {
            'objective': 'reg:squarederror',
            'eval_metric': 'rmse',
            'tree_method': 'hist',
            'device': 'cuda',
            'learning_rate': trial.suggest_float('learning_rate', lr_range[0], lr_range[1], log=True),
            'max_depth': trial.suggest_int('max_depth', depth_range[0], depth_range[1]),
            'min_child_weight': trial.suggest_int('min_child_weight', 50, 300),
            'subsample': trial.suggest_float('subsample', 0.6, 1.0),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
            'reg_lambda': trial.suggest_float('reg_lambda', lambda_range[0], lambda_range[1], log=True),
            'seed': 42
        }
        
        model = xgb.train(
            params, dtrain_tune, num_boost_round=400,
            evals=[(dvalid_tune, 'valid')],
            early_stopping_rounds=40, verbose_eval=False
        )
        return model.best_score

    study = optuna.create_study(direction='minimize')
    study.optimize(objective, n_trials=n_trials)
    
    del dtrain_tune, dvalid_tune; gc.collect()
    return study.best_params

params_short = run_optuna_for_horizon(day_idx=2, depth_range=(6, 10), lr_range=(0.03, 0.15), lambda_range=(0.1, 3.0), n_trials=5)
params_medium = run_optuna_for_horizon(day_idx=7, depth_range=(4, 7), lr_range=(0.01, 0.1), lambda_range=(3.0, 15.0), n_trials=5)
params_long = run_optuna_for_horizon(day_idx=14, depth_range=(3, 5), lr_range=(0.005, 0.05), lambda_range=(10.0, 50.0), n_trials=5)

# 🔥 IN KẾT QUẢ BỘ THAM SỐ TỐI ƯU
print("\n🌟 BỘ THAM SỐ TỐI ƯU 3 CHẶNG ĐÃ TÌM THẤY:")
print(f"👉 Ngắn hạn (1-3): {params_short}")
print(f"👉 Trung hạn (4-10): {params_medium}")
print(f"👉 Dài hạn (11-16): {params_long}")

# ---------------------------------------------------------
# 6. VÒNG LẶP TRAIN 16 NGÀY KẾT HỢP SEED AVERAGING
# ---------------------------------------------------------
all_test_predictions = []
seeds = [42, 2026, 999]

for i in tqdm(range(1, 17), desc="🔥 Train 16 Models (Seed Averaging)"):
    print(f"\n⏳ Đang huấn luyện Ngày {i}/16...")
    
    if i <= 3: 
        current_params = params_short.copy()
        max_iters = 1000; patience = 50
    elif i <= 10: 
        current_params = params_medium.copy()
        max_iters = 1200; patience = 50 
    else: 
        current_params = params_long.copy()
        max_iters = 1500; patience = 75 # Đã ép xung chặng dài hạn

    current_params.update({
        'objective': 'reg:squarederror',
        'eval_metric': 'rmse',
        'tree_method': 'hist',
        'device': 'cuda'
    })

    features_for_model = base_features + safe_cat_features + [f'promo_future_{i}']
    
    target_train = df_train[f'target_{i}'].fillna(0).astype('float32')
    target_valid = df_valid[f'target_{i}'].fillna(0).astype('float32')
    
    dtrain = xgb.DMatrix(df_train[features_for_model], label=target_train, weight=weight_train, enable_categorical=True)
    dvalid = xgb.DMatrix(df_valid[features_for_model], label=target_valid, weight=weight_valid, enable_categorical=True)
    dtest = xgb.DMatrix(X_test_base[features_for_model], enable_categorical=True)
    
    preds_day_i = np.zeros(len(X_test_base))
    
    for s in seeds:
        current_params['seed'] = s
        
        model = xgb.train(
            current_params, dtrain, num_boost_round=max_iters,
            evals=[(dvalid, 'valid')],
            early_stopping_rounds=patience, verbose_eval=False
        )
        
        pred_s = np.expm1(np.clip(model.predict(dtest, iteration_range=(0, model.best_iteration + 1)), 0, 10))
        preds_day_i += pred_s / len(seeds)
    
    del dtrain, dvalid, target_train, target_valid; gc.collect()

    pred_df = X_test_base[['store_nbr', 'item_nbr']].copy()
    pred_df['date'] = pd.to_datetime('2017-08-15') + pd.Timedelta(days=i)
    pred_df['unit_sales'] = preds_day_i
    all_test_predictions.append(pred_df)
    
    del dtest, pred_df; gc.collect()

🚀 CELL 3: TIỀN XỬ LÝ, TUNING OPTUNA & HUẤN LUYỆN SEED AVERAGING...
   -> Đang dọn dẹp Inf và NaN...
   -> Cắt ranh giới Validation (16/07 - 30/07)...
   -> 🔍 Số feature sẽ đưa vào Model: 58 (Kho vũ khí đã sẵn sàng)

🔍 BẮT ĐẦU TÌM KIẾM THAM SỐ TỐI ƯU BẰNG OPTUNA (5 Trials/Chặng để tiết kiệm thời gian)...

🎯 Đang Tune cho đại diện Ngày 2...


[I 2026-04-28 11:05:11,495] A new study created in memory with name: no-name-e6c6d714-1b72-4cb8-8056-9a06166a0ee0
[I 2026-04-28 11:10:48,765] Trial 0 finished with value: 0.5710356987360972 and parameters: {'learning_rate': 0.09473397187578259, 'max_depth': 8, 'min_child_weight': 237, 'subsample': 0.7740052920522202, 'colsample_bytree': 0.7007189231468876, 'reg_lambda': 0.34121195940645793}. Best is trial 0 with value: 0.5710356987360972.
[I 2026-04-28 11:17:10,880] Trial 1 finished with value: 0.5724758759581764 and parameters: {'learning_rate': 0.033873377860111555, 'max_depth': 8, 'min_child_weight': 193, 'subsample': 0.9838957421296008, 'colsample_bytree': 0.889772613983203, 'reg_lambda': 0.34068180512227425}. Best is trial 0 with value: 0.5710356987360972.
[I 2026-04-28 11:23:36,122] Trial 2 finished with value: 0.5717009422022928 and parameters: {'learning_rate': 0.03365039473897244, 'max_depth': 8, 'min_child_weight': 272, 'subsample': 0.8691769542118246, 'colsample_bytree': 0.7


🎯 Đang Tune cho đại diện Ngày 7...


[I 2026-04-28 11:31:37,374] A new study created in memory with name: no-name-46830988-999d-4dae-b7a0-86829d1affcc
[I 2026-04-28 11:35:51,338] Trial 0 finished with value: 0.6011159020982055 and parameters: {'learning_rate': 0.06069118458473435, 'max_depth': 4, 'min_child_weight': 129, 'subsample': 0.6154181476470754, 'colsample_bytree': 0.7928385634633872, 'reg_lambda': 5.038839526724399}. Best is trial 0 with value: 0.6011159020982055.
[I 2026-04-28 11:41:09,336] Trial 1 finished with value: 0.5995929410835109 and parameters: {'learning_rate': 0.021150258789595817, 'max_depth': 5, 'min_child_weight': 161, 'subsample': 0.8935434554416497, 'colsample_bytree': 0.754152646336502, 'reg_lambda': 10.891486581794739}. Best is trial 1 with value: 0.5995929410835109.
[I 2026-04-28 11:44:09,202] Trial 2 finished with value: 0.5982779566853674 and parameters: {'learning_rate': 0.0642815785038905, 'max_depth': 6, 'min_child_weight': 279, 'subsample': 0.7654498135171298, 'colsample_bytree': 0.83025


🎯 Đang Tune cho đại diện Ngày 14...


[I 2026-04-28 11:55:15,125] A new study created in memory with name: no-name-dd774da4-6f22-4897-85ed-ad1428127052
[I 2026-04-28 12:00:24,495] Trial 0 finished with value: 0.6174997871381546 and parameters: {'learning_rate': 0.027358580205103275, 'max_depth': 5, 'min_child_weight': 288, 'subsample': 0.9469657042793155, 'colsample_bytree': 0.7627147554155592, 'reg_lambda': 13.826416070870772}. Best is trial 0 with value: 0.6174997871381546.
[I 2026-04-28 12:05:04,233] Trial 1 finished with value: 0.6199386474736399 and parameters: {'learning_rate': 0.024778569813187025, 'max_depth': 4, 'min_child_weight': 91, 'subsample': 0.7670305383932432, 'colsample_bytree': 0.9185383194885127, 'reg_lambda': 11.19793903253556}. Best is trial 0 with value: 0.6174997871381546.
[I 2026-04-28 12:09:11,238] Trial 2 finished with value: 0.6223479886109502 and parameters: {'learning_rate': 0.03409165853281936, 'max_depth': 3, 'min_child_weight': 202, 'subsample': 0.7201955676398608, 'colsample_bytree': 0.839


🌟 BỘ THAM SỐ TỐI ƯU 3 CHẶNG ĐÃ TÌM THẤY:
👉 Ngắn hạn (1-3): {'learning_rate': 0.09473397187578259, 'max_depth': 8, 'min_child_weight': 237, 'subsample': 0.7740052920522202, 'colsample_bytree': 0.7007189231468876, 'reg_lambda': 0.34121195940645793}
👉 Trung hạn (4-10): {'learning_rate': 0.0642815785038905, 'max_depth': 6, 'min_child_weight': 279, 'subsample': 0.7654498135171298, 'colsample_bytree': 0.8302574063345124, 'reg_lambda': 10.377047724852375}
👉 Dài hạn (11-16): {'learning_rate': 0.027358580205103275, 'max_depth': 5, 'min_child_weight': 288, 'subsample': 0.9469657042793155, 'colsample_bytree': 0.7627147554155592, 'reg_lambda': 13.826416070870772}


🔥 Train 16 Models (Seed Averaging):   0%|          | 0/16 [00:00<?, ?it/s]


⏳ Đang huấn luyện Ngày 1/16...

⏳ Đang huấn luyện Ngày 2/16...

⏳ Đang huấn luyện Ngày 3/16...

⏳ Đang huấn luyện Ngày 4/16...

⏳ Đang huấn luyện Ngày 5/16...

⏳ Đang huấn luyện Ngày 6/16...

⏳ Đang huấn luyện Ngày 7/16...

⏳ Đang huấn luyện Ngày 8/16...

⏳ Đang huấn luyện Ngày 9/16...

⏳ Đang huấn luyện Ngày 10/16...

⏳ Đang huấn luyện Ngày 11/16...

⏳ Đang huấn luyện Ngày 12/16...

⏳ Đang huấn luyện Ngày 13/16...

⏳ Đang huấn luyện Ngày 14/16...

⏳ Đang huấn luyện Ngày 15/16...

⏳ Đang huấn luyện Ngày 16/16...


In [4]:
# CELL 4
print("🚀 CELL 4: XỬ LÝ POST-PROCESSING & XUẤT FILE XGBOOST...")
final_pred_df = pd.concat(all_test_predictions, ignore_index=True)

test_raw = pd.read_parquet(f'{BASE}/test_clean.parquet')
test_raw['date']           = pd.to_datetime(test_raw['date'])
final_pred_df['date']      = pd.to_datetime(final_pred_df['date'])

test_raw['store_nbr']      = test_raw['store_nbr'].astype(str)
final_pred_df['store_nbr'] = final_pred_df['store_nbr'].astype(str)

test_raw['item_nbr']       = test_raw['item_nbr'].astype(float).fillna(-1).astype(int)
final_pred_df['item_nbr']  = final_pred_df['item_nbr'].astype(int)

submission = test_raw[['id', 'store_nbr', 'item_nbr', 'date']].merge(
    final_pred_df, on=['store_nbr', 'item_nbr', 'date'], how='left')

# ☠️ TRẢM MẶT HÀNG NGỪNG KINH DOANH
if 'days_since_last_sale' in X_test_base.columns:
    print("   -> ☠️ Đang gán 0 cho các mặt hàng ngừng kinh doanh (>= 14 ngày)...")
    dead_items = X_test_base[X_test_base['days_since_last_sale'] >= 14][['store_nbr', 'item_nbr']].copy()
    dead_items['store_nbr'] = dead_items['store_nbr'].astype(str) 
    
    dead_mask = submission.set_index(['store_nbr', 'item_nbr']).index.isin(dead_items.set_index(['store_nbr', 'item_nbr']).index)
    submission.loc[dead_mask, 'unit_sales'] = 0

submission['unit_sales'] = submission['unit_sales'].fillna(0)

assert submission['id'].duplicated().sum() == 0, "❌ DUPLICATE ID!"

OUTPUT_FILE = '/kaggle/working/submission_xgboost_grandmaster.csv'
submission[['id', 'unit_sales']].to_csv(OUTPUT_FILE, index=False)
print(f"🏆 Đã lưu thành công file Grandmaster tại: {OUTPUT_FILE}")

🚀 CELL 4: XỬ LÝ POST-PROCESSING & XUẤT FILE XGBOOST...
   -> ☠️ Đang gán 0 cho các mặt hàng ngừng kinh doanh (>= 14 ngày)...
🏆 Đã lưu thành công file Grandmaster tại: /kaggle/working/submission_xgboost_grandmaster.csv
